Packages Download

In [65]:
!uv pip install --upgrade openai openai-agents pypdf python-dotenv requests gradio pandas

Resolved 75 packages in 253ms                                        
Checked 75 packages in 1ms


In [66]:
from dotenv import load_dotenv
import os

from openai import AsyncAzureOpenAI
from agents import (
    Agent,
    Runner,
    function_tool,
    OpenAIChatCompletionsModel,
    set_tracing_disabled,
    SQLiteSession,
    handoff,
)
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX
from pypdf import PdfReader
import gradio as gr
import pandas as pd

API Key Checkover.

In [67]:
load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT = "gpt-5.4-ptu"

print("Endpoint loaded:", bool(AZURE_OPENAI_ENDPOINT))
print("API key loaded:", bool(AZURE_OPENAI_API_KEY))

Endpoint loaded: True
API key loaded: True


Disabling Tracer.

In [68]:
set_tracing_disabled(True)

azure_client = AsyncAzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-10-21",
)

azure_model = OpenAIChatCompletionsModel(
    model=DEPLOYMENT,
    openai_client=azure_client,
)

print("Azure model configured for the OpenAI Agents SDK.")

Azure model configured for the OpenAI Agents SDK.


Monthly Expenses collection Tool.

In [69]:
def read_pdf(file_path: str) -> str:
    reader = PdfReader(file_path)
    text = ""

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    return text


# AI Agent Tool
@function_tool
def extract_pdf_text(file_path: str) -> str:
    """Extract text content from a PDF file."""
    return read_pdf(file_path)


# PDF Collection
MONTHLY_EXPENSES_COLLECTION = {
    "monthly_expenses_pdf_1": {
        "filename": "MonthlySpendingData_Placeholder.pdf",
        "content": read_pdf(
            "data/MonthlySpendingData_Placeholder.pdf"
        )
    }
}


spending_data_text = MONTHLY_EXPENSES_COLLECTION[
    "monthly_expenses_pdf_1"
]["content"]

print("Loaded PDF text length:", len(spending_data_text), "characters")

Loaded PDF text length: 275 characters


Here, our agent will have a csv parsing tool.
Paint points: different banks have different csv format. The sample csv is a placeholder

In [70]:
@function_tool
def parse_csv_transactions_tool(file_path: str) -> dict:
    """Parse and validate transactions from a CSV file."""
    try:
        df = pd.read_csv(file_path)

        required_columns = ["date", "description", "amount"]
        missing = [col for col in required_columns if col not in df.columns]

        if missing:
            return {"success": False, "error": f"Missing columns: {missing}"}

        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

        invalid_rows = df[["date", "amount"]].isna().any(axis=1)
        if invalid_rows.any():
            return {
                "success": False,
                "error": f"{invalid_rows.sum()} rows contain invalid dates or amounts"
            }

        return {
            "success": True,
            "transaction_count": len(df),
            "transactions": df.assign(
                date=df["date"].dt.strftime("%Y-%m-%d")
            ).to_dict(orient="records")
        }

    except (OSError, pd.errors.ParserError, UnicodeError) as e:
        return {"success": False, "error": str(e)}

A tool that allows AI to calculate spending summary

In [71]:
@function_tool
def calculate_spending_summary_tool(file_path: str) -> dict:
    """Calculate total expenses and spending by category from a CSV file."""
    df = pd.read_csv(file_path)

    required_columns = {"description", "amount"}
    if not required_columns.issubset(df.columns):
        return {"error": "Missing required columns"}

    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

    if df["amount"].isna().any():
        return {"error": "Invalid transaction amounts"}

    # Example categorization based on merchant descriptions
    def categorize(description: str) -> str:
        description = description.lower()

        if any(word in description for word in ["walmart", "kroger", "aldi"]):
            return "Groceries"
        elif any(word in description for word in ["starbucks", "mcdonald", "chipotle"]):
            return "Dining"
        elif any(word in description for word in ["shell", "exxon", "uber"]):
            return "Transportation"
        else:
            return "Other"

    df["category"] = df["description"].fillna("").apply(categorize)

    spending_by_category = (
        df.groupby("category")["amount"].sum().round(2).to_dict()
    )

    return {
        "total_spending": round(float(df["amount"].sum()), 2),
        "transaction_count": len(df),
        "spending_by_category": spending_by_category,
    }

In [72]:
@function_tool
def save_transactions_tool(
    file_path: str,
    output_path: str = "data/saved_transactions.json"
) -> dict:
    """Save financial transactions from a CSV file into a JSON file."""

    df = pd.read_csv(file_path)

    required_columns = {"date", "description", "amount"}
    if not required_columns.issubset(df.columns):
        return {"error": "Missing required columns"}

    if df[list(required_columns)].isna().any().any():
        return {"error": "Transactions contain missing values"}

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

    if df[["date", "amount"]].isna().any().any():
        return {"error": "Transactions contain invalid dates or amounts"}

    transactions = df.assign(
        date=df["date"].dt.strftime("%Y-%m-%d")
    ).to_dict(orient="records")

    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)

    with output.open("w", encoding="utf-8") as f:
        json.dump(transactions, f, indent=4)

    return {
        "status": "success",
        "saved_transactions": len(transactions),
        "file_path": str(output),
    }

Tool Prompt.

In [77]:
instructions = f"""{RECOMMENDED_PROMPT_PREFIX}

You are Expense Analyst Agent, You are a Financial Advizing Agent that Analyzes financial data provided by users through 
CSV files, chat inputs, PDF files. You categorizes transactions, identifies spending patterns, and summarizes expenses for the user


You are to, when prompted by the user, to list the monthly expenses provided in the pdf from the MONTHLY_EXPENSES_COLLECTION and tell them how they could better optimize their spending.

. 

"""

ExpenseAnalystAgent = Agent(
    name="ExpenseAnalystAgent",
    handoff_description="Analyzes budget and summarizes spending.",
    instructions=instructions,
    tools=[extract_pdf_text, parse_csv_transactions_tool, calculate_spending_summary_tool, save_transactions_tool],
    model=azure_model,
)

print("Agent built:", ExpenseAnalystAgent.name)

#when executed with tools that can read the MONTHLY_EXPENSES_COLLECTION, the chatbot should be able to properly assess the file
#for now, its a pdf as a placeholder but we can change that later. 
# We will add more to the description of the agent we are creating the tools - Chris


Agent built: ExpenseAnalystAgent


Test agent

In [78]:
result = await Runner.run(
    ExpenseAnalystAgent,
    "Read data/MonthlySpendingData_Placeholder.pdf and summarize my expenses."
)

print(result.final_output)

Here’s a summary of your monthly expenses from the PDF:

### Monthly Expenses
- Rent: **$1,000**
- Gas: **$200**
- Groceries: **$400**
- Spotify: **$10**
- Netflix: **$15**
- Take-out: **$50**
- Car Payment: **$160**
- Car Insurance: **$100**
- Medical Insurance: **$300**
- Amazon Orders: **$100**
- Miscellaneous: **$400**

### Total Spending
- **$3,500**

### Income / Take-Home
- **$765**

### Quick Analysis
Your expenses are much higher than your reported take-home amount. That suggests one of these may be true:
- the income figure is incomplete,
- the spending total includes non-monthly items,
- or you are currently spending beyond your means.

### Biggest Spending Categories
Your largest expenses are:
1. **Rent: $1,000**
2. **Groceries: $400**
3. **Miscellaneous: $400**
4. **Medical Insurance: $300**
5. **Gas: $200**

### Ways to Optimize Spending
- **Review “Miscellaneous” ($400):** This is a large flexible category and the best place to start cutting.
- **Reduce grocery costs:** 

In [79]:
result = await Runner.run(
    ExpenseAnalystAgent,
    "What is my spending by category?"
)

print(result.final_output)

I can help with that. Please provide your transaction data first, for example by:

- uploading a CSV file of your transactions, or
- sharing the file path to the CSV/PDF you want analyzed, or
- pasting the transaction data here.

Once I have the data, I’ll summarize your spending by category and point out optimization opportunities.


CHAT WITH GRADIO!

In [80]:
chat_session = SQLiteSession("gradio_user", "user_expenses.db")

async def chat(message, history):
    result = await Runner.run(ExpenseAnalystAgent, message, session=chat_session)
    return result.final_output

gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [76]:
# Hey guys, so this is basically the skeleton of the Expenses Agent, I formatted it in a way that it will read a single pdf file when given the tools to do so
#We will end up adding memory as well, preferably after the tools are implemented for this agent
#It may be a little thrown together in terms of the structure, but I made what I could on my own. If you guys have issues with it, we will modify it together. 
#-Chris